# Movies Dataset — Analyze Phase

Compute the findings that answer the business question: which movie characteristics — budget,
genre, runtime, release timing — are most associated with financial return (ROI) and audience
rating. Source: `data/processed/movies_clean.parquet` and `data/processed/movie_genres.parquet`
(from the Process phase). Small aggregate tables are saved to `data/summary/` for reuse in the
Share phase.

## Setup

In [1]:
import pandas as pd
import numpy as np
import os

PROC = "../data/processed"
SUMMARY = "../data/summary"
os.makedirs(SUMMARY, exist_ok=True)

movies = pd.read_parquet(os.path.join(PROC, "movies_clean.parquet"))
genres = pd.read_parquet(os.path.join(PROC, "movie_genres.parquet"))
print(f"movies: {movies.shape}, genres: {genres.shape}")

# vote_average is only meaningful where at least one vote was recorded
movies["vote_average_valid"] = movies["vote_average"].where(movies["vote_count"] > 0)
n_valid_votes = (movies["vote_count"] > 0).sum()
print(f"Rows with vote_count > 0: {n_valid_votes}")

movies: (45430, 18), genres: (91006, 3)
Rows with vote_count > 0: 42534


## A. Correlations — does budget predict return or rating?

Pearson correlation between the movie-level financial/quality metrics.

In [2]:
pairs = [
    ("budget", "revenue"),
    ("budget", "roi"),
    ("budget", "vote_average_valid"),
    ("runtime", "vote_average_valid"),
    ("vote_count", "vote_average_valid"),
    ("roi", "vote_average_valid"),
    ("popularity", "vote_average_valid"),
]
corr_rows = []
for a, b in pairs:
    sub = movies[[a, b]].dropna()
    r = sub[a].corr(sub[b])
    corr_rows.append({"var_a": a, "var_b": b, "r": round(r, 4), "n": len(sub)})
    print(f"{a} vs {b}: r={r:.4f}, n={len(sub)}")

corr_df = pd.DataFrame(corr_rows)
corr_df.to_csv(os.path.join(SUMMARY, "correlations.csv"), index=False)

budget vs revenue: r=0.7303, n=5375
budget vs roi: r=-0.0254, n=5307
budget vs vote_average_valid: r=0.0758, n=8790
runtime vs vote_average_valid: r=0.1095, n=42376
vote_count vs vote_average_valid: r=0.1224, n=42534
roi vs vote_average_valid: r=0.0077, n=5303
popularity vs vote_average_valid: r=0.0984, n=42534


## B. Genre-level return and rating

Joins the exploded movie x genre table back to the movie-level metrics. Two separate sample-size
floors: rating stats require >=100 movies in the genre (rating data is dense — 42,534 of 45,430
movies have a vote_count > 0); ROI stats require >=20 movies **with a valid roi** specifically
(roi data is sparse — only 5,307 movies total), since a genre with a large `n_movies` can still
have almost no usable roi data (e.g. TV Movie: 765 movies, only 1 with a valid roi).

In [3]:
genre_movies = genres.merge(
    movies[["id", "budget", "revenue", "roi", "vote_average_valid", "vote_count"]], on="id", how="left"
)

g_all = genre_movies.groupby("genre").agg(
    n_movies=("id", "count"),
    n_with_roi=("roi", "count"),
    avg_roi=("roi", "mean"),
    median_roi=("roi", "median"),
    avg_vote_average=("vote_average_valid", "mean"),
    avg_budget=("budget", "mean"),
).reset_index()

genre_rating = g_all[g_all["n_movies"] >= 100].sort_values("avg_vote_average", ascending=False)
print("--- Genre ranked by avg rating (n_movies >= 100) ---")
print(genre_rating[["genre", "n_movies", "avg_vote_average"]].round(3).to_string(index=False))

genre_roi = g_all[g_all["n_with_roi"] >= 20].sort_values("median_roi", ascending=False)
print("\n--- Genre ranked by median ROI (n_with_roi >= 20) ---")
print(genre_roi[["genre", "n_with_roi", "avg_roi", "median_roi"]].round(3).to_string(index=False))

g_all.to_csv(os.path.join(SUMMARY, "genre_stats.csv"), index=False)
genre_rating.to_csv(os.path.join(SUMMARY, "genre_rating.csv"), index=False)
genre_roi.to_csv(os.path.join(SUMMARY, "genre_roi.csv"), index=False)

--- Genre ranked by avg rating (n_movies >= 100) ---
          genre  n_movies  avg_vote_average
    Documentary      3930             6.652
      Animation      1930             6.446
        History      1398             6.411
          Music      1597             6.332
            War      1322             6.283
          Drama     20243             6.172
          Crime      4304             6.097
        Romance      6730             6.031
        Foreign      1619             5.976
         Comedy     13176             5.966
        Mystery      2464             5.960
         Family      2767             5.934
        Fantasy      2309             5.919
      Adventure      3490             5.873
         Action      6590             5.745
       Thriller      7618             5.734
        Western      1042             5.710
       TV Movie       765             5.642
Science Fiction      3042             5.467
         Horror      4670             5.311

--- Genre ranked by me

Note: `avg_roi` is heavily skewed by a handful of extreme low-budget breakout hits (e.g.
*Paranormal Activity*, ROI ~ 12,889x, in the Prepare/Process phase). `median_roi` (on the
`n_with_roi >= 20` genres) is the more reliable ranking metric — it's what the Share-phase genre
ROI chart uses.

## C. Budget tier vs. rating and ROI

Buckets movies into 5 budget tiers to test the classic claim: does a bigger budget buy a better
movie (higher rating), or just a bigger swing on return?

In [4]:
bins = [0, 5_000_000, 20_000_000, 50_000_000, 100_000_000, np.inf]
labels = ["<$5M", "$5-20M", "$20-50M", "$50-100M", "$100M+"]
movies["budget_tier"] = pd.cut(movies["budget"], bins=bins, labels=labels)

tier = movies.groupby("budget_tier", observed=True).agg(
    n_movies=("id", "count"),
    avg_vote_average=("vote_average_valid", "mean"),
    avg_roi=("roi", "mean"),
    median_roi=("roi", "median"),
).reindex(labels)
print(tier.round(3))
tier.to_csv(os.path.join(SUMMARY, "budget_tier.csv"))

             n_movies  avg_vote_average  avg_roi  median_roi
budget_tier                                                 
<$5M             3714             5.984   27.168       1.929
$5-20M           2562             6.117    2.490       0.776
$20-50M          1556             6.118    1.545       0.720
$50-100M          718             6.140    1.528       0.990
$100M+            330             6.407    2.119       1.773


## D. Runtime vs. rating

Buckets movies into runtime bands to check whether length associates with rating.

In [5]:
rbins = [0, 80, 100, 120, 150, np.inf]
rlabels = ["<80min", "80-100min", "100-120min", "120-150min", "150min+"]
movies["runtime_tier"] = pd.cut(movies["runtime"].where(movies["runtime"] > 0), bins=rbins, labels=rlabels)

rt = movies.groupby("runtime_tier", observed=True).agg(
    n_movies=("id", "count"),
    avg_vote_average=("vote_average_valid", "mean"),
).reindex(rlabels)
print(rt.round(3))
rt.to_csv(os.path.join(SUMMARY, "runtime_tier.csv"))

              n_movies  avg_vote_average
runtime_tier                            
<80min            6791             6.099
80-100min        20878             5.755
100-120min       10943             6.195
120-150min        3610             6.493
150min+           1393             6.643


## E. Release-month seasonality in revenue

Average and median revenue by release month, among movies with a known revenue — a signal for
release-timing strategy.

In [6]:
mv_rev = movies.dropna(subset=["revenue", "release_month"])
month_stats = mv_rev.groupby("release_month").agg(
    n_movies=("id", "count"),
    avg_revenue=("revenue", "mean"),
    median_revenue=("revenue", "median"),
).reindex(range(1, 13))
print(month_stats.round(0))
month_stats.to_csv(os.path.join(SUMMARY, "release_month.csv"))

               n_movies  avg_revenue  median_revenue
release_month                                       
1                   512   31401516.0       9424636.0
2                   519   52025178.0      16756372.0
3                   564   66200403.0      17989870.0
4                   558   62062921.0      14702850.0
5                   585   98536546.0      13417292.0
6                   596  121037730.0      30000000.0
7                   567   93067215.0      31554855.0
8                   658   45464569.0      15043094.0
9                   904   32696293.0       7531282.0
10                  697   48089356.0      12902790.0
11                  541   95887328.0      27100027.0
12                  696   96173285.0      35726000.0


## Key findings

1. **Budget strongly predicts revenue (r = 0.73)** but is essentially uncorrelated with ROI
   (r = -0.03) — a bigger budget buys a bigger box office in absolute dollars, but not a better
   *return* on the money spent.
2. **Budget barely predicts audience rating (r = 0.08)**, and profitability doesn't predict
   rating either (ROI vs. rating, r = 0.01) — a movie's financial success and its critical
   reception are close to independent of each other in this data.
3. **The genres that pay off financially are mostly not the genres that rate best.** By median
   ROI (genres with >=20 movies with usable roi data): Animation leads (1.77x), then Horror
   (1.49x) and Family (1.48x). By average rating (genres with >=100 movies): Documentary leads
   (6.65), then Animation (6.45) and History (6.41). **Animation is the rare double-winner** —
   top-3 on both rating and ROI — but also one of the most expensive genres to produce on average
   ($46.7M). **Horror is the clearest "reliable-return, low-prestige" genre**: 2nd-highest median
   ROI at the *lowest* average budget of any major genre ($9.3M), but dead last on rating (5.31).
   Documentary's rating lead is worth flagging with caution — only 56 of 3,930 documentaries have
   usable ROI data, too thin to assess its return with confidence.
4. **The <$5M budget tier has the highest median ROI (1.93x)**, roughly double the $5-100M
   "mid-budget" range (0.72-0.99x) — consistent with the film-industry pattern where low-budget
   films need only a modest absolute gross to post a large percentage return, while mid-budget
   films carry real box-office risk without a blockbuster's marketing/distribution edge.
5. **Longer movies skew modestly higher-rated**: average rating rises from 5.76 (80-100 min) to
   6.64 (150min+) — a real but weak pattern (consistent with the r=0.11 runtime/rating
   correlation), not something to chase on its own.
6. **Revenue is seasonal**: June ($121M avg) and July ($93M avg) — the summer blockbuster window
   — and November/December ($96M avg) — the holiday window — clearly outperform January and
   September (both under $33M avg), the industry's well-known "dump months."

## Interpretation

For the client's greenlighting decision: **budget size buys box-office scale, not quality or
percentage return** — a large budget is a bet on absolute dollars, not a safer or better-reviewed
film. The strongest, most actionable signals are genre-specific ROI patterns and release-timing:
genre should inform the *return* case, release-window should inform the *revenue* case, and
neither of those two levers meaningfully predicts audience rating, which should be evaluated on
its own terms (creative/critical fit) rather than assumed to follow from budget or genre choice.